# Phase 3: Preprocess v2 (Training-Grade)

This notebook runs the upgraded preprocessing pipeline and validates:
- Sinhala-priority cleaning with conservative short-text handling
- aggressive exact + near-duplicate filtering
- source + time aware unseen holdout split
- preservation of already labeled data

In [5]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import os

REPO_ROOT = Path.cwd().resolve()
for p in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (p / 'data_collection').exists():
        REPO_ROOT = p
        break
DATA_ROOT = Path('/root/separate_volume')
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT
RUN_ID = pd.Timestamp.now().strftime('preprocess_%Y%m%d_%H%M%S')

REPO_ROOT, DATA_ROOT


WindowsPath('D:/client-projects/sl-social-media-risk-analysis')

In [6]:
env = dict(**os.environ)
env['PYTHONPATH'] = str(REPO_ROOT)

cleaned_output = DATA_ROOT / 'datasets/preprocessing/runs' / RUN_ID / 'final_cleaned_dataset.csv'
cleaned_summary = DATA_ROOT / 'datasets/preprocessing/runs' / RUN_ID / 'final_cleaned_dataset_summary.json'
workflow_output_dir = DATA_ROOT / 'annotation/workflow/runs' / RUN_ID
split_output_dir = DATA_ROOT / 'datasets/splits/runs' / RUN_ID

elakiri_path = DATA_ROOT / 'datasets/sources/elakiri_comments.csv'
gossip_path = DATA_ROOT / 'datasets/sources/gossip_lanka_comments.csv'
youtube_path = DATA_ROOT / 'datasets/sources/youtube_comments.csv'

cmd_clean = [
    'python', '-m', 'data_collection.pipelines.build_final_cleaned_dataset',
    '--elakiri-path', str(elakiri_path),
    '--gossip-path', str(gossip_path),
    '--youtube-path', str(youtube_path),
    '--sinhala-threshold', '0.2',
    '--min-text-chars', '8',
    '--max-text-chars', '1200',
    '--output-csv', str(cleaned_output),
    '--summary-path', str(cleaned_summary),
]
print('Running:', ' '.join(cmd_clean))
subprocess.check_call(cmd_clean, cwd=str(REPO_ROOT), env=env)

cmd_refresh = [
    'python', '-m', 'data_collection.pipelines.refresh_labeling_with_locked_holdout',
    '--input-csv', str(cleaned_output),
    '--output-dir', str(workflow_output_dir),
    '--split-dir', str(split_output_dir),
    '--holdout-ratio', '0.15',
    '--existing-holdout-csv', str(DATA_ROOT / 'datasets/splits/current/locked_unseen_holdout.csv'),
    '--preserve-existing-holdout',
]
existing_labels = DATA_ROOT / 'datasets/labeled/annotator_a_llm.csv'
if existing_labels.exists():
    cmd_refresh.extend(['--existing-label-csv', str(existing_labels)])

print('Running:', ' '.join(cmd_refresh))
subprocess.check_call(cmd_refresh, cwd=str(REPO_ROOT), env=env)


Running: powershell -ExecutionPolicy Bypass -File D:\client-projects\sl-social-media-risk-analysis\scripts\run-preprocess-v2.ps1 -ProjectRoot D:\client-projects\sl-social-media-risk-analysis -RunId preprocess_20260316_231840 -SinhalaThreshold 0.2 -MinTextChars 8 -MaxTextChars 1200 -HoldoutRatio 0.15


CalledProcessError: Command '['powershell', '-ExecutionPolicy', 'Bypass', '-File', 'D:\\client-projects\\sl-social-media-risk-analysis\\scripts\\run-preprocess-v2.ps1', '-ProjectRoot', 'D:\\client-projects\\sl-social-media-risk-analysis', '-RunId', 'preprocess_20260316_231840', '-SinhalaThreshold', '0.2', '-MinTextChars', '8', '-MaxTextChars', '1200', '-HoldoutRatio', '0.15']' returned non-zero exit status 4294770688.

In [ ]:
clean_summary_path = DATA_ROOT / 'datasets' / 'preprocessing' / 'runs' / RUN_ID / 'final_cleaned_dataset_summary.json'
workflow_summary_path = DATA_ROOT / 'annotation' / 'workflow' / 'runs' / RUN_ID / 'workflow_summary.json'

clean_summary = json.loads(clean_summary_path.read_text(encoding='utf-8'))
workflow_summary = json.loads(workflow_summary_path.read_text(encoding='utf-8'))

clean_summary, workflow_summary['counts']


In [ ]:
final_df = pd.read_csv(DATA_ROOT / 'datasets' / 'preprocessing' / 'runs' / RUN_ID / 'final_cleaned_dataset.csv')
train_dev_df = pd.read_csv(DATA_ROOT / 'datasets' / 'splits' / 'runs' / RUN_ID / 'train_dev_pool.csv')
holdout_df = pd.read_csv(DATA_ROOT / 'datasets' / 'splits' / 'runs' / RUN_ID / 'locked_unseen_holdout.csv')

print('final_cleaned_dataset rows:', len(final_df))
print('train_dev_pool rows:', len(train_dev_df))
print('locked_unseen_holdout rows:', len(holdout_df))

print('\nSource distribution (final):')
print(final_df['source'].value_counts())

print('\nSource distribution (holdout):')
print(holdout_df['source'].value_counts())

if 'time_bucket' in holdout_df.columns:
    print('\nHoldout time buckets (top 12):')
    print(holdout_df['time_bucket'].value_counts().head(12))
